# Final Analytical Notebook — Business KPI Analysis

This notebook analyzes the illustrative sample dataset used in the Interactive Business Dashboard. The figures are for portfolio demonstration only and are not verified company results.


## 1. Load libraries and dataset
The CSV below is embedded so the notebook can run without a separate file. Replace it with validated business data for real decisions.


In [ ]:
from io import StringIO
import pandas as pd
import matplotlib.pyplot as plt

csv_text = '''month,region,category,revenue,orders,new_customers,acquisition_spend,customers_at_start,customers_lost
2026-01-01,India,Electronics,125000,250,100,15000,1200,36
2026-02-01,India,Clothing,118000,280,110,16500,1264,32
2026-03-01,USA,Electronics,145000,290,125,20000,1320,40
2026-04-01,UK,Home,132000,300,120,18000,1405,35
2026-05-01,India,Home,155000,320,135,20250,1490,30
2026-06-01,USA,Clothing,162000,340,140,21000,1580,38
2026-07-01,Australia,Electronics,170000,350,150,22500,1682,34
2026-08-01,UK,Clothing,165000,360,145,21750,1798,36
2026-09-01,Canada,Home,185000,380,160,24000,1907,32
2026-10-01,India,Electronics,195000,400,175,26250,2035,35
2026-11-01,UAE,Clothing,210000,420,185,27750,2175,30
2026-12-01,USA,Home,230000,450,200,30000,2325,34
'''
df = pd.read_csv(StringIO(csv_text), parse_dates=['month'])
df.head()


## 2. Data quality checks


In [ ]:
required = ['month','region','category','revenue','orders','new_customers',
            'acquisition_spend','customers_at_start','customers_lost']
print('Rows:', len(df))
print('Missing values by column:\n', df[required].isna().sum())
print('Duplicate rows:', df.duplicated().sum())
df[required].describe(include='all')


## 3. Executive KPIs


In [ ]:
total_revenue = df['revenue'].sum()
total_orders = df['orders'].sum()
total_new_customers = df['new_customers'].sum()
total_acquisition_spend = df['acquisition_spend'].sum()
weighted_cac = total_acquisition_spend / total_new_customers if total_new_customers else 0
weighted_aov = total_revenue / total_orders if total_orders else 0
simple_churn_proxy = df['customers_lost'].sum() / df['customers_at_start'].sum() * 100 if df['customers_at_start'].sum() else 0
kpis = pd.DataFrame({'Metric':['Revenue','Orders','New customers','Acquisition spend','Weighted CAC','Weighted AOV','Simple churn proxy (%)'],
                     'Value':[total_revenue,total_orders,total_new_customers,total_acquisition_spend,weighted_cac,weighted_aov,simple_churn_proxy]})
kpis


## 4. Revenue trend


In [ ]:
monthly = df.groupby('month', as_index=False)['revenue'].sum().sort_values('month')
monthly.plot(x='month', y='revenue', kind='line', marker='o', legend=False, title='Monthly Revenue')
plt.ylabel('Revenue ($)'); plt.grid(alpha=0.25); plt.show()
endpoint_growth = (monthly['revenue'].iloc[-1] / monthly['revenue'].iloc[0] - 1) * 100
print(f'Endpoint growth: {endpoint_growth:.1f}%')


## 5. Category and region summaries


In [ ]:
category_summary = df.groupby('category', as_index=False)['revenue'].sum().sort_values('revenue', ascending=False)
region_summary = df.groupby('region', as_index=False)['revenue'].sum().sort_values('revenue', ascending=False)
display(category_summary)
display(region_summary)
category_summary.plot(x='category', y='revenue', kind='bar', legend=False, title='Revenue by Category')
plt.ylabel('Revenue ($)'); plt.tight_layout(); plt.show()


## 6. Acquisition efficiency


In [ ]:
df['monthly_cac'] = df['acquisition_spend'] / df['new_customers'].replace(0, pd.NA)
df[['month','acquisition_spend','new_customers','monthly_cac']]


## 7. Strategic interpretation and limitations

- Revenue rises across the endpoints in the sample; this does not establish why it rose or whether the trend will continue.
- Category and region totals are confounded because each month has one region/category row. Avoid causal conclusions from these totals.
- The churn figure is a simplified proxy because monthly starting-customer counts are summed; use unique customer cohorts for a proper retention analysis.
- The ROI scenario in the executive report is illustrative and must be replaced with validated margin, cost, and uplift assumptions.

## 8. Recommended next steps
1. Validate data and define metric ownership.
2. Add margin, channel, discounts, and customer-cohort data.
3. Run controlled acquisition and retention pilots.
4. Review incremental contribution at 30, 60, and 90 days.
